# GPT-SoVITS WebUI

## Env Setup (Run Once Only)
## 环境配置, 只需运行一次

### 1.

In [ ]:
%%writefile /content/setup.sh
set -e

cd /content

git clone https://github.com/RVC-Boss/GPT-SoVITS.git

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS

pip install ipykernel

bash install.sh --device CU126 --source HF --download-uvr5

### 2.

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh

## Launch WebUI
## 启动 WebUI

### starlette バージョン固定（gradio互換性エラー対策）
GPT-SoVITS の `webui.py` は `GPTSoVITS` という専用の conda 環境（python3.10）の中で動く。
ここで `source activate GPTSoVITS &&` を付けずに `pip install` すると、base の Colab カーネル（python3.12）に
インストールされてしまい、実際にサーバーが読み込む starlette には反映されない。

根本原因: gradio の `routes.py` が旧式の `templates.TemplateResponse(name, context)` という
位置引数の呼び出しをしており、新しい Starlette（0.46 以降）ではこの呼び出し方がサポートされず
`TypeError: unhashable type: 'dict'` がページ読み込みのたびに発生する。
参考: https://github.com/RVC-Boss/GPT-SoVITS/issues/2762

In [ ]:
!source activate GPTSoVITS && pip install "starlette>=0.40.0,<0.46.0"

In [ ]:
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && python webui.py

## データセット準備（Google Drive からダウンロード）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, glob, os
DATA = '/content/dataset'
SRC = '/content/drive/MyDrive/99_未分類/gpt_sovits_data_20260703'
os.makedirs(DATA, exist_ok=True)
for z in glob.glob(f'{SRC}/*.zip'):
    zipfile.ZipFile(z).extractall(DATA)

# train.list の相対パスを絶対パスへ書き換え
for lst in glob.glob(f'{DATA}/*/train.list'):
    root = os.path.dirname(lst)
    rows = open(lst, encoding='utf-8').read().splitlines()
    rows = [f'{root}/' + r if r.startswith('wavs/') else r for r in rows]
    open(lst.replace('train.list', 'train_abs.list'), 'w', encoding='utf-8').write('\n'.join(rows) + '\n')
print('done:', glob.glob(f'{DATA}/*/train_abs.list'))